In [ ]:
# Lecture 19 — Writing Data: modes, partitionBy, controlling file count
# Spark 101 · Module 6 · Lecture 19
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Some data to write, spread across a few partitions --------------------
import os

df = spark.createDataFrame(
    [
        (1, "Ada",   "ES", 19.99),
        (2, "Linus", "FI", 5.50),
        (3, "Grace", "US", 1200.00),
        (4, "Alan",  "ES", 42.00),
        (5, "Edsger","US", 8.75),
    ],
    ["id", "name", "country", "price"],
).repartition(3)   # pretend we arrived here with 3 partitions

print("partitions:", df.rdd.getNumPartitions())
base = "/tmp/spark101_l19"

In [ ]:
# --- Save mode: overwrite. Writing is an ACTION — this runs the plan. -------
out = f"file:{base}/plain"
df.write.mode("overwrite").parquet(out)

# One partition -> one file. 3 partitions -> 3 part-*.parquet files.
files = [f for f in os.listdir(f"{base}/plain") if f.endswith(".parquet")]
print("part files written:", len(files))

In [ ]:
# --- append vs overwrite ---------------------------------------------------
df.write.mode("append").parquet(out)     # adds more files alongside
print("after append:", len([f for f in os.listdir(f"{base}/plain") if f.endswith(".parquet")]))

df.write.mode("overwrite").parquet(out)  # wipes and rewrites
print("after overwrite:", len([f for f in os.listdir(f"{base}/plain") if f.endswith(".parquet")]))

In [ ]:
# --- Control the file count: coalesce(1) -> a single file ------------------
# Fine for small results; DANGEROUS on big data (funnels all rows to 1 Worker).
one = f"file:{base}/single"
df.coalesce(1).write.mode("overwrite").parquet(one)
print("single-file dir:", [f for f in os.listdir(f"{base}/single") if f.endswith(".parquet")])

In [ ]:
# --- partitionBy: one FOLDER per column value ------------------------------
part = f"file:{base}/by_country"
df.write.mode("overwrite").partitionBy("country").parquet(part)

# Spark writes country=ES/, country=FI/, country=US/ — data split on disk.
print("top-level folders:", sorted(os.listdir(f"{base}/by_country")))

In [ ]:
# --- The payoff: a filtered read prunes whole folders ----------------------
# Spark reads only country=ES/, never opens FI/ or US/.
es = spark.read.parquet(part).filter("country = 'ES'")
es.show()
es.explain()   # look for a PartitionFilters / pushed partition filter on country

In [ ]:
# --- CSV write needs header; compression is an option too ------------------
df.write.mode("overwrite").option("header", True).csv(f"file:{base}/csv_out")
df.write.mode("overwrite").option("compression", "gzip").parquet(f"file:{base}/gz_out")
print("done")

In [ ]:
# Write other formats: JSON and CSV (with header).
df.write.mode("overwrite").json(f"file:{base}/as_json")
df.write.mode("overwrite").option("header", True).csv(f"file:{base}/as_csv")
print("wrote json + csv")

In [ ]:
# partitionBy MULTIPLE columns -> nested folders (country=.../year=...).
from pyspark.sql.functions import lit
(df.withColumn("year", lit(2026))
   .write.mode("overwrite").partitionBy("country", "year")
   .parquet(f"file:{base}/by_country_year"))
print(sorted(os.listdir(f"{base}/by_country_year")))

In [ ]:
# append mode adds data alongside what's already there.
df.write.mode("append").parquet(f"file:{base}/plain")
print("rows after append:", spark.read.parquet(f"file:{base}/plain").count())

In [ ]:
# saveAsTable: a managed table, queryable by name in SQL.
df.write.mode("overwrite").saveAsTable("spark101_l19_people")
spark.sql("SELECT country, count(*) AS n FROM spark101_l19_people GROUP BY country").show()

In [ ]:
# Recap:
#   - df.write is an ACTION — it runs the whole pipeline.
#   - Always set mode() explicitly: overwrite / append / errorifexists / ignore.
#   - File count = partition count; coalesce/repartition to control it.
#   - partitionBy writes folder-per-value -> partition pruning on read.
# Next lecture: Parquet vs CSV — which format to actually write.